# Actividad: Agente Reactivo Simple

En esta actividad implementaremos y evaluaremos un **agente reactivo simple**.

Un agente reactivo simple selecciona una acción únicamente a partir del estado actual:

$\pi : \mathcal{S} \rightarrow \mathcal{A}$

donde:

- $\mathcal{S}$ es el conjunto de estados observables.
- $\mathcal{A}$ es el conjunto de acciones.
- El agente no conserva memoria de estados anteriores.
- El agente no realiza planificación.

El objetivo de la actividad es:

1. Implementar un agente reactivo mediante una política tabular.
2. Implementar un agente aleatorio como referencia.
3. Ejecutar episodios de interacción agente-entorno.
4. Comparar experimentalmente el desempeño de ambos agentes.

In [2]:
# Importar bibliotecas necesarias
import random

## Entorno: GridWorld

Utilizaremos un entorno discreto representado mediante una cuadrícula de tamaño `size × size`.

### Componentes del entorno

- **Estado:** posición actual del agente `(x, y)`.
- **Estado inicial:** definido por `start`.
- **Estado meta:** definido por `goal`.
- **Acciones posibles:** `UP`, `DOWN`, `LEFT`, `RIGHT`.

El espacio de estados puede representarse como:

$\mathcal{S} = \{(x,y) \mid 0 \leq x,y < size\}$

y el conjunto de acciones como:

$\mathcal{A} = \{UP, DOWN, LEFT, RIGHT\}$

El entorno es determinista: una acción válida produce siempre el movimiento correspondiente.

La función de transición puede representarse como:

$\mathcal{T} : \mathcal{S} \times \mathcal{A} \rightarrow \mathcal{S}$

### Métodos principales

- `reset()` reinicia el entorno al estado inicial.
- `perceive()` devuelve las acciones válidas desde el estado actual.
- `step(action)` ejecuta una acción y actualiza el estado.
- `is_done()` verifica si el agente alcanzó la meta.

In [22]:
# -----------------------------
# Entorno
# -----------------------------
class GridWorld:
    def __init__(self, size=3, start=(0, 0), goal=(2, 2)):
        self.size = size
        self.start = start
        self.goal = goal
        self.agent_pos = start

# Métodos principales
    def reset(self):
        self.agent_pos = self.start
        return self.agent_pos

    def perceive(self):
        # posición del agente
        x,y = self.agent_pos
        valid_actions = []

        if x > 0:
            valid_actions.append("LEFT")
        
        if x < self.size -1:
            valid_actions.append("RIGHT")
        
        if y > 0:
            valid_actions.append("DOWN")

        if y < self.size -1:
            valid_actions.append("UP")

        return valid_actions

    def step(self, action):
        # posición del agente
        x,y = self.agent_pos

        if action == "UP":
            self.agent_pos = (x, y+1)
        
        elif action == "DOWN":
            self.agent_pos = (x, y-1)

        if action == "LEFT":
            self.agent_pos = (x-1, y)

        elif action == "RIGHT":
            self.agent_pos = (x+1, y)

    def is_done(self):
        if self.agent_pos == self.goal:
            return True
        else:
            return False

## Ejecución de un episodio

Para evaluar un agente necesitamos ejecutar su interacción con el entorno durante varios pasos.

En cada instante:

1. El agente observa el estado actual.
2. Selecciona una acción.
3. El entorno ejecuta la acción.
4. Se actualiza el estado.
5. Se verifica si se alcanzó la meta.

Un episodio termina cuando:

- el agente alcanza la meta, o
- se alcanza un número máximo de pasos.

El límite de pasos evita ejecuciones indefinidas cuando una política no logra alcanzar la meta.

La función `run_episode()` devuelve:

- `success`: indica si el agente alcanzó la meta.
- `steps`: número de acciones ejecutadas.
- `trajectory`: secuencia de estados visitados.

In [27]:
def run_episode(env, agent, max_steps=20):
    env.reset()

    trajectory = [env.agent_pos]

    for step in range(max_steps):

        if env.is_done():
            return True, step, trajectory
        
        action = agent.act()

        if action is None:
            return False, step, trajectory
        
        env.step(action)
        trajectory.append(env.agent_pos)

    return env.is_done(), max_steps, trajectory

## Agente Reactivo Simple

Representaremos la política del agente mediante un diccionario.

Un agente reactivo simple implementa una política:

$\pi : \mathcal{S} \rightarrow \mathcal{A}$

de modo que a cada estado $s$ se le asigna directamente una acción $a$:

$\pi(s) = a$

El agente:

- observa únicamente el estado actual;
- consulta la acción asociada con ese estado;
- no conserva memoria;
- no realiza búsqueda;
- no analiza estados futuros.

La política contiene, por tanto, conocimiento previamente especificado sobre cómo debe actuar el agente en cada estado.

In [7]:
# -----------------------------
# Agente Reactivo Simple
# -----------------------------
class SimpleReactiveAgent:
    def __init__(self, env, policy):
        self.env = env
        self.policy = policy
    
    def act(self):
        if self.env.is_done():
            return None
        
        state = self.env.agent_pos
        action = self.policy.get(state, None)

        return action

## Política tabular

Definiremos ahora una política para un entorno de tamaño `3 × 3`, con:

- estado inicial `(0, 0)`;
- meta `(2, 2)`.

Cada entrada del diccionario representa una regla de la forma:

`estado → acción`

Por ejemplo:

`(0, 0) → RIGHT`

significa que cuando el agente se encuentre en el estado `(0, 0)`, ejecutará la acción `RIGHT`.

In [19]:
# Política determinista π : S → A
policy = {
    (0, 0): "RIGHT",
    (0, 1): "RIGHT",
    (0, 2): "RIGHT",

    (1, 0): "RIGHT",
    (1, 1): "RIGHT",
    (1, 2): "RIGHT",

    (2, 0): "UP",
    (2, 1): "UP",

    (2, 2): None
}

## Prueba del agente reactivo

Ejecutaremos un episodio y observaremos:

- si el agente alcanza la meta;
- cuántos pasos utiliza;
- qué trayectoria sigue.

In [32]:
env_reactive = GridWorld(
    size=3,
    start=(0, 0),
    goal=(2, 2)
)

reactive_agent = SimpleReactiveAgent(
    env_reactive,
    policy
)

success, steps, trajectory = run_episode(
    env_reactive,
    reactive_agent,
    max_steps=20
)

print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)

Éxito: True
Pasos: 4
Trayectoria: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]


## Análisis de la trayectoria

Desde `(0,0)` hasta `(2,2)`, la distancia mínima en una cuadrícula sin obstáculos es:

$d^* = |x_g-x_0| + |y_g-y_0|$

En este caso:

$d^* = |2-0| + |2-0| = 4$

Compara este valor con el número de pasos utilizado por el agente.

Observa que el agente puede seguir una trayectoria óptima aun cuando **no realiza búsqueda ni planificación**.

La trayectoria óptima está implícitamente codificada en la política.

## Agente Aleatorio

Utilizaremos ahora un agente aleatorio como línea base de comparación.

Este agente:

1. obtiene las acciones válidas en el estado actual;
2. selecciona una de ellas al azar.

Si existen $k$ acciones válidas, cada acción tiene probabilidad:

$P(a \mid s) = \frac{1}{k}$

El agente aleatorio:

- no mantiene memoria;
- no realiza planificación;
- no utiliza conocimiento sobre la posición de la meta.

In [34]:
# -----------------------------
# Agente Aleatorio
# -----------------------------
class RandomAgent:
    def __init__(self, env):
        self.env = env
    
    def act(self):
        if self.env.is_done():
            return None
        
        state = self.env.agent_pos
        action = random.choice(self.env.perceive())
    
        return action

## Prueba del agente aleatorio

Ejecuta varias veces la siguiente celda.

Observa que el agente puede seguir trayectorias diferentes debido a que la selección de acciones es aleatoria.

In [49]:
env_random = GridWorld(
    size=3,
    start=(0, 0),
    goal=(2, 2)
)

random_agent = RandomAgent(env_random)

success, steps, trajectory = run_episode(
    env_random,
    random_agent,
    max_steps=20
)

print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)

Éxito: True
Pasos: 4
Trayectoria: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]


## Evaluación experimental

Una sola ejecución no es suficiente para comparar agentes, especialmente cuando alguno de ellos tiene comportamiento aleatorio.

Ejecutaremos cada agente durante múltiples episodios y calcularemos dos métricas.

### Tasa de éxito

La tasa de éxito representa la proporción de episodios en los que el agente alcanza la meta:

$\text{Tasa de éxito} =
\frac{\text{episodios exitosos}}
{\text{episodios totales}}$

### Número promedio de pasos

Para los episodios exitosos calcularemos:

$\overline{L} =
\frac{1}{N_s}
\sum_{i=1}^{N_s} L_i$

donde:

- $N_s$ es el número de episodios exitosos;
- $L_i$ es el número de pasos utilizado en el episodio $i$.

Un buen agente debería alcanzar la meta con alta frecuencia y utilizar pocos pasos.

In [50]:
def evaluate_agent(agent_type, episodes=100, max_steps=20):
    successes = 0
    successful_steps = []

    for _ in range(episodes):

        env = GridWorld(
            size=3,
            start=(0, 0),
            goal=(2, 2)
        )

        if agent_type == "reactive":
            agent = SimpleReactiveAgent(env, policy)

        elif agent_type == "random":
            agent = RandomAgent(env)

        else:
            raise ValueError("Tipo de agente no válido")

        success, steps, _ = run_episode(
            env,
            agent,
            max_steps
        )

        if success:
            successes += 1
            successful_steps.append(steps)

    success_rate = successes / episodes

    if successful_steps:
        avg_steps = sum(successful_steps) / len(successful_steps)
    else:
        avg_steps = None

    return success_rate, avg_steps

## Comparación de los agentes

Ejecutaremos ambos agentes bajo las mismas condiciones:

- mismo tamaño del entorno;
- mismo estado inicial;
- misma meta;
- mismo número máximo de pasos.

El agente reactivo es determinista, por lo que esperamos que repita el mismo comportamiento en todos los episodios.

El agente aleatorio puede producir resultados distintos entre ejecuciones.

In [66]:
reactive_success, reactive_steps = evaluate_agent(
    "reactive",
    episodes=100,
    max_steps=20
)

random_success, random_steps = evaluate_agent(
    "random",
    episodes=100,
    max_steps=20
)

print("Agente reactivo")
print("Tasa de éxito:", reactive_success)
print("Pasos promedio:", reactive_steps)

print("\nAgente aleatorio")
print("Tasa de éxito:", random_success)
print("Pasos promedio:", random_steps)

Agente reactivo
Tasa de éxito: 1.0
Pasos promedio: 4.0

Agente aleatorio
Tasa de éxito: 0.75
Pasos promedio: 10.693333333333333


In [52]:
print(f"{'Agente':<15} {'Éxito':<12} {'Pasos promedio'}")
print("-" * 45)

print(
    f"{'Reactivo':<15} "
    f"{reactive_success:<12.2f} "
    f"{reactive_steps:.2f}"
)

print(
    f"{'Aleatorio':<15} "
    f"{random_success:<12.2f} "
    f"{random_steps:.2f}"
)

Agente          Éxito        Pasos promedio
---------------------------------------------
Reactivo        1.00         4.00
Aleatorio       0.76         10.45


## Preguntas de reflexión

1. ¿Por qué el agente reactivo obtiene siempre el mismo resultado?

    Porque el agente reactivo tiene una política establecida que le permite conocer la trayectoria más corta para llegar a la meta para este GridWorld.

2. ¿Por qué el agente aleatorio puede producir trayectorias diferentes?

    Porque el agente aleatorio, con base en la percepción que recibe del entorno, selecciona sus acciones al azar entre una lista de acciones válidas.

3. ¿Alguno de los dos agentes realiza búsqueda en el espacio de estados?

    No. Esto porque el agente reactivo utiliza su política para decidir directamente sus acciones, mientras que el agente aleatorio selecciona una acción al azar.

4. ¿Por qué el agente reactivo puede encontrar una trayectoria óptima sin utilizar un algoritmo de búsqueda?

    Porque su entorno, en este caso el GridWorld, es observable y determinista, con una meta fija, lo que permite omitir el uso de un algoritmo de búsqueda y plantear una solución que sabemos que va a funcionar siempre.

5. ¿Qué conocimiento sobre el problema está implícitamente codificado en la política tabular?

    La trayectoria óptima para llegar a la meta.

6. ¿Qué ocurriría si cambiamos la posición de la meta pero mantenemos la misma política?

    En ese caso, el agente solo llegaría a la meta si la posición de dicha meta se encuentre dentro de la trayectoria definida en la política.

7. ¿La política tabular utilizada puede considerarse general para cualquier entorno `GridWorld`?

    No. Esto porque la política tabular se define a partir de un tamaño específico del GridWorld y una meta fija, por lo que si el tamaño cambia, y la meta no es la misma o no es fija en cada iteración, esta política no podría considerarse general.

8. ¿Qué ventaja ofrece el agente reactivo frente al agente aleatorio?

    Que para entornos observables y deterministas, donde la política tabular le permite llegar a la meta siempre, el agente reactivo obtiene una tasa de éxito mejor y requiere menos pasos que el agente aleatorio.

9. ¿Qué limitación importante presenta una política tabular cuando aumenta el número de estados?

    Que la política tabular puede crecer desmedidamente hasta que sea imposible de sostenerse, dado que tendría que almacenar demasiadas posibilidades de estado-acción.

## Reto

Genera de forma pseudoaleatoria un estado inicial y una meta distintos dentro del `GridWorld`.

Después, diseña una política reactiva que permita al agente alcanzar siempre la meta desde cualquier estado inicial.

### Condiciones

- El estado inicial y la meta deben ser diferentes.
- No debes construir manualmente una tabla para cada combinación posible.
- La acción debe depender únicamente del estado actual y de la meta.

### Preguntas

1. ¿Qué reglas utilizaste para decidir el movimiento?

    Las reglas utilizadas para decidir el movimiento del agente fueron las siguientes:
* Comparar la posición actual (x,y) con la posición de la meta (gx, gy).
* Si x es menor que gx, moverse a la derecha (siempre que esté dentro de las acciones válidas del entorno).
* Si x es mayor que gx, movserse a la izquierda (siempre que esté dentro de las acciones válidas del entorno).
* Si y es menor que gy, moverse hacia arriba (siempre que esté dentro de las acciones válidas del entorno).
* Si y es mayor que gy, moverse hacia abajo (siempre que esté dentro de las acciones válidas del entorno).

    De este modo siempre busca acercarse a la meta mediante la comparación de coordenadas.

2. ¿El agente realiza búsqueda o planificación?

    El agente no realiza búsqueda al no explicar posibles estados o caminos y tampoco realiza planificación al no determinar una secuencia de acciones antes de la ejecución. En cambio, lo que hace es depender únicamente del estado actual y en base a las reglas establecidas en la definicón del agentes decide su siguiente curso de acción.

3. ¿Por qué esta política funciona para cualquier par inicio--meta en este entorno?

    Porque las reglas no dependen de una trayectoria específica, dado que utilizan las coordenadas de la posición actual del agente y de la meta, lo que le permite adaptarse a diferentes posiciones. De este modo, mientre el inicio y la meta estén dentro de GridWorld, estas reglas pueden determinar hacia donde moverse.

In [68]:
# Generar inicio y meta pseudoaleatorios

size = 3

start = (
    random.randrange(size),
    random.randrange(size)
)

goal = start

while goal == start:
    goal = (
        random.randrange(size),
        random.randrange(size)
    )

print("Inicio:", start)
print("Meta:", goal)

Inicio: (2, 1)
Meta: (1, 0)


In [1]:
class GeneralReactiveAgent:
    def __init__(self, env):
        self.env = env

    def act(self):
        x, y = self.env.agent_pos
        gx, gy = self.env.goal

        if self.env.is_done():
            return None

        # Completar las reglas
        if (x < gx) and "RIGHT" in self.env.perceive():
            action = "RIGHT"
        elif (x > gx) and "LEFT" in self.env.perceive():
            action = "LEFT"
        elif (y < gy) and "UP" in self.env.perceive():
            action = "UP"
        elif (y > gy) and "DOWN" in self.env.perceive():
            action = "DOWN"

        return None


In [ ]:
# Vázquez Villegas Carlos Roberto
# UNAM FI
# Inteligencia Artficial

# Prueba el agente reactivo general

general_size = 3

general_start = (
    random.randrange(general_size),
    random.randrange(general_size)
)

general_goal = general_start

while general_goal == general_start:
    general_goal = (
        random.randrange(general_size),
        random.randrange(general_size)
    )

print("Inicio:", general_start)
print("Meta:", general_goal)

env_general = GridWorld(
    general_size,
    general_start,
    general_goal
)

general_agent = GeneralReactiveAgent(env_general)

gen_success, gen_steps, gen_trajectory = run_episode(
    env_general,
    general_agent,
    max_steps=20
)

print("Éxito:", gen_success)
print("Pasos:", gen_steps)
print("Trayectoria:", gen_trajectory)


Inicio: (0, 2)
Meta: (0, 0)
Éxito: True
Pasos: 2
Trayectoria: [(0, 2), (0, 1), (0, 0)]
